# Simulating every trial, end to end

The full per-trial pipeline: take the force a participant produced on each trial, drive the
motoneuron pool with it, and measure the firing rates that come out. This replaces the
hand-edited loops in `Loop_run_sims.ipynb` and `frs.ipynb`, which carried lists of trial
numbers that had to be uncommented one session at a time.

**The chain**

| step | where | output |
|---|---|---|
| 1. Scale the recorded force | `experiments_data_analysis.ipynb` §4.2 | `Scaled_force/<subject>_alltrials/DayXX_trialYY.npy` |
| 2. Simulate each trial | **§5 below** | `DR_<subject>_alltrials/DayXX/DR_<subject>_DayXX_trialYY.jld2` |
| 3. Firing rates per trial | **§7 below** | `firings/<subject>/DayXX_trialYY_rates.jld2` |
| 4. Population GLM | `generalized_linear_model_analysis.ipynb` §5 | pseudo-R² per session |

Step 1 has already been run — this notebook starts from the scaled traces on disk. Force is
scaled on the Python side and never rescaled here; every output stores its own sampling
rate, so nothing downstream has to assume one.

**Expect this to take a while.** One twenty-second trial is about 420,000 integration steps
across the whole pool, roughly eight minutes, and writes a 70 MB raster. All 130 trials is
on the order of 17 hours and 9 GB. Both §5 and §7 are resumable and can be run a session at
a time.

---

## 1. Packages and model code

In [ ]:
# `using` first -- creation_model.jl needs @unpack at include time.
using NPZ, Interpolations, Distributions, Random, JLD2, Printf, Statistics
using Parameters, DSP, SymDict, Match, CSV, DataFrames

# Every path below is relative to the repository root, so step up if the kernel
# was started inside notebooks/.
basename(pwd()) == "notebooks" && cd("..")
println("working directory: ", pwd())

folder = "utils/functions_run_simulations/"
for f in sort(readdir(folder))
    endswith(f, ".jl") && include(joinpath(folder, f))
end

println("✓ Model code loaded")

---

## 2. Configuration

`FORCE_DIR` is where the scaled traces live and `OUT_ROOT` is where the rasters go. The
layout of both is what the GLM notebook expects: force as `DayXX_trialYY.npy` in one flat
folder, spikes as `DayXX/` subfolders, with the day and trial recoverable from each filename.

`FORCE_FS` is the sampling rate of the recorded force, 2048 Hz. `EXTRA_GAIN` multiplies the
scaled trace on the way in; leave it at 1.0 to run exactly the amplitude that was saved. It
is written into every output either way, so a gain applied here stays visible downstream.

In [ ]:
SUBJECT    = "S1"
FORCE_FS   = 2048.0      # sampling rate of the scaled force traces (Hz)
EXTRA_GAIN = 1.0         # 1.0 = run the amplitude that was saved

VTHCA_FILE = "Data_MN_Model/Data_final/Subjects_optim_process/VthCa_TA_subject1_gamma1.jld2"
FORCE_DIR  = "Data_MN_Model/Data_final/Scaled_force/$(SUBJECT)_alltrials"
OUT_ROOT   = "Data_MN_Model/Data_final/Spikes/Simulations_with_force_as_input/DR_$(SUBJECT)_alltrials"

println("subject : ", SUBJECT)
println("force   : ", FORCE_DIR)
println("output  : ", OUT_ROOT)

isdir(FORCE_DIR) || @warn "No scaled force found. Run scale_and_save_all_trials_mean in experiments_data_analysis.ipynb first."

---

## 3. Build the pool

One pool, built once, used for every trial. `seed` fixes the rheobase jitter so repeated runs
use the same pool and results stay comparable between gains and scaling choices.

`VthCa` comes from the per-unit fit in `optimization_vthca_values.ipynb`, and its length sets
the size of the pool.

In [ ]:
MNs, nr_s = build_pool(VTHCA_FILE)
println("✓ Pool built: ", nr_s, " motoneurons")

---

## 4. What is about to run

Lists the trials found and how many already have a raster on disk. Writes nothing.

In [ ]:
# Trials are discovered from the force filenames: DayXX_trialYY.npy
force_files = sort([f for f in readdir(FORCE_DIR) if endswith(f, ".npy")])

trials = [(parse(Int, m[1]), parse(Int, m[2]), joinpath(FORCE_DIR, f))
          for f in force_files
          for m in [match(r"Day(\d+)_trial(\d+)", f)] if m !== nothing]

outfile_for(day, trial) = joinpath(OUT_ROOT, @sprintf("Day%02d", day),
                                   @sprintf("DR_%s_Day%02d_trial%02d.jld2", SUBJECT, day, trial))

days_found = sort(unique(first.(trials)))
done_already = count(isfile(outfile_for(d, t)) for (d, t, _) in trials)

@printf("%d trials across %d sessions\n", length(trials), length(days_found))
@printf("  already simulated : %d / %d\n", done_already, length(trials))
@printf("  still to run      : %d\n", length(trials) - done_already)

println("\ntrials per session:")
for d in days_found
    @printf("  Day%02d : %2d trials\n", d, count(x -> x[1] == d, trials))
end

---

## 5. Run the sweep

One simulation per trial. Each force trace is resampled to the 1 kHz grid the integrator
expects, run through the pool, and saved with the provenance of the force that produced it.

**This is a long run.** The pool is integrated at a fixed 0.05 ms step for the full duration
of every trial, so a twenty-second trial is roughly four hundred thousand steps across the
whole pool. A full session is tens of minutes; all sessions is hours.

It is resumable: trials whose output already exists are skipped, so an interrupted sweep
restarts by re-running this cell. Set `OVERWRITE = true` to recompute them instead.

Start with a single session — set `DAYS_TO_RUN = [1]` — and check the output before
committing to the whole sweep.

In [ ]:
DAYS_TO_RUN = nothing    # e.g. [1] for one session; nothing = every session
OVERWRITE   = false      # true = recompute trials that already have output

selected = DAYS_TO_RUN === nothing ? trials : filter(x -> x[1] in DAYS_TO_RUN, trials)
written  = String[]

@printf("Running %d trials, extra_gain = %.2f, output fs = %.0f Hz\n\n",
        length(selected), EXTRA_GAIN, SPIKE_FS_HZ)

for (day, trial, force_path) in selected
    outfile = outfile_for(day, trial)
    mkpath(dirname(outfile))

    if !OVERWRITE && isfile(outfile)
        println("  skip (exists): ", basename(outfile))
        push!(written, outfile)
        continue
    end

    force = npzread(force_path)
    trace_ms, duration_s = resample_to_ms(force, FORCE_FS)
    trace_ms = trace_ms .* EXTRA_GAIN

    @printf("  Day%02d trial%02d  %.1fs  peak=%.3f ... ", day, trial, duration_s, maximum(trace_ms))
    flush(stdout)
    elapsed = @elapsed spike_array = simulate_trace(trace_ms)
    @printf("done in %.0fs (%d spikes)\n", elapsed, Int(sum(spike_array)))
    flush(stdout)

    fs         = SPIKE_FS_HZ
    force_file = force_path
    gain       = EXTRA_GAIN
    subject    = SUBJECT

    JLD2.@save outfile spike_array fs force_file gain subject day trial
    push!(written, outfile)
end

println("\n✓ ", length(written), " files under ", OUT_ROOT)

---

## 6. Check what was written

Counts the rasters on disk against the trials found, then reads one back to confirm it
carries its sampling rate and provenance.

In [ ]:
per_day = [(d, begin
        dir = joinpath(OUT_ROOT, @sprintf("Day%02d", d))
        isdir(dir) ? count(f -> endswith(f, ".jld2"), readdir(dir)) : 0
    end) for d in days_found]

println("rasters written per session:")
for (d, n) in per_day
    @printf("  Day%02d : %2d / %2d\n", d, n, count(x -> x[1] == d, trials))
end
@printf("\ntotal: %d / %d trials simulated\n", sum(last.(per_day)), length(trials))

existing = [outfile_for(d, t) for (d, t, _) in trials if isfile(outfile_for(d, t))]

if !isempty(existing)
    JLD2.@load first(existing) spike_array fs subject day trial force_file gain
    println("\nexample: ", basename(first(existing)))
    @printf("  spike_array : %d motoneurons x %d samples\n", size(spike_array)...)
    @printf("  fs          : %.0f Hz\n", fs)
    @printf("  provenance  : %s Day%02d trial%02d, gain=%.2f\n", subject, day, trial, gain)
    @printf("  force       : %s\n", basename(force_file))
    @printf("  spikes      : %d\n", Int(sum(spike_array)))
else
    println("\nNothing written yet -- run the sweep above.")
end

---

## 7. Firing rates from the rasters

The measurement the simulation exists to produce. For each motoneuron, the average firing
rate is `1 / mean(ISI)` over the trial — the same estimator used on the experimental side,
including the 80 Hz cap that discards units whose intervals imply implausible rates.

One file of rates per trial, so a session can be pooled or a trial inspected on its own.
This is what the firing-rate histograms are built from.

In [ ]:
RATES_DIR = "Data_MN_Model/Data_final/firings/$(SUBJECT)"
FR_CAP    = 80.0      # Hz; rates above this are discarded, as on the experimental side

mkpath(RATES_DIR)

"""Average firing rate per motoneuron, 1 / mean(ISI), for one raster."""
function average_rates(spike_array, fs; cap = FR_CAP)
    rates = Float64[]
    for mn in 1:size(spike_array, 1)
        spike_pos = findall(==(1.0), spike_array[mn, :])
        length(spike_pos) > 1 || continue
        isis = diff(spike_pos) ./ fs                 # seconds
        r = 1 / mean(isis)                           # Hz
        r <= cap && push!(rates, r)
    end
    return rates
end

rate_files = String[]
for (day, trial, _) in trials
    raster = outfile_for(day, trial)
    isfile(raster) || continue

    out = joinpath(RATES_DIR, @sprintf("Day%02d_trial%02d_rates.jld2", day, trial))
    if isfile(out)
        push!(rate_files, out)
        continue
    end

    JLD2.@load raster spike_array fs
    avg_rates_per_neuron = average_rates(spike_array, fs)
    JLD2.@save out avg_rates_per_neuron fs subject=SUBJECT day trial
    push!(rate_files, out)
    @printf("  Day%02d trial%02d : %3d units, median %.1f Hz\n",
            day, trial, length(avg_rates_per_neuron), median(avg_rates_per_neuron))
end

@printf("\n%d rate files under %s\n", length(rate_files), RATES_DIR)

### 7.1 Firing-rate distributions

One panel per trial of a session, so the spread can be compared against the experimental
distributions. `check_firing_rates_online_control.ipynb` does the same comparison against
the decomposed recordings directly.

In [ ]:
using Plots

DAY_TO_PLOT = first(days_found)

session = [(t, joinpath(RATES_DIR, @sprintf("Day%02d_trial%02d_rates.jld2", d, t)))
           for (d, t, _) in trials if d == DAY_TO_PLOT]
session = [(t, f) for (t, f) in session if isfile(f)]

if isempty(session)
    println("No rate files for Day", DAY_TO_PLOT, " yet -- run section 7 first.")
else
    ncol = 5
    nrow = ceil(Int, length(session) / ncol)
    p = Plots.plot(layout = (nrow, ncol), size = (400 * ncol, 260 * nrow),
                   legend = false, grid = false)

    for (i, (t, f)) in enumerate(session)
        JLD2.@load f avg_rates_per_neuron
        Plots.histogram!(p, avg_rates_per_neuron, bins = 20, subplot = i,
                         xlabel = "Firing rate (Hz)", ylabel = "Count",
                         title = @sprintf("Day%02d trial%02d", DAY_TO_PLOT, t),
                         xlim = (0, FR_CAP))
    end
    display(p)
end

---

## 8. What reads this

- `generalized_linear_model_analysis.ipynb` §5 scans `OUT_ROOT` for `DayXX/` rasters and
  pairs each with the force trace of the same `DayXX_trialYY` — which is why both sides keep
  that naming. Point `SIM_SPIKE_DIR` at `OUT_ROOT` and `SIM_FORCE_DIR` at `FORCE_DIR`.
- `check_firing_rates_online_control.ipynb` compares these rasters against the decomposed
  recordings trial by trial, rather than only their rate distributions.